# Image segmentation and discrete structures
<p>
    
__Part 1__: Image formation and thresholding
    
__Quantitative Big Imaging__ ETHZ: 227-0966-00L
    
</p>
    
<div class="rows">
    <div class="column23">
        <p style="font-size:1em;">March 12, 2026</p>
        <br /><br />
        <p style="font-size:1.5em;padding-bottom: 0.25em;">Anders Kaestner</p>  
        <p style="font-size:1em;">Laboratory for Neutron Scattering and Imaging<br />Paul Scherrer Institut</p>
    </div>
    <div class="column13">
        <img src="../../docs/figures/np_segmentation_4159870_000000.svg" style="height:300px" />
    </div>
</div>   

## Today's lecture

- Motivation
- Qualitative Approaches
- Image formation and interpretation problems
- Thresholding
    - Other types of images
    - Selecting a good threshold
- Implementation
- Morphological image processing
- Partial volume effects

### Load some modules

In [ ]:
from skimage.io          import imread
from skimage.color       import rgb2gray
import matplotlib.pyplot as plt
from skimage.morphology  import disk
from scipy.ndimage       import zoom
import numpy             as np
import pandas as pd
from skimage.morphology import ball
import tifffile as tiff
import plotsupport as ps

%matplotlib inline

# For the 3D rendering
# import plotly.offline as py
# from plotly.figure_factory import create_trisurf
from skimage.measure import marching_cubes

## Applications

In this lecture we are going to focus on basic segmentation approaches that work well for simple two-phase materials. Segmenting complex samples like  
- Beyond 1 channel of depth
- Multiple phase materials
- Filling holes in materials
- Segmenting Fossils
- Attempting to segment the cortex in brain imaging (see figure below)

can be a very challenging task. Such tasks will be covered in later lectures.

```{figure} figures/cortex.png
:width: 8cm
An x-ray CT slice of the cortex.
```

<table>
    <tr>
        <td>
        
- Simple two-phase materials (bone, cells, etc)
- Beyond 1 channel of depth
    - Multiple phase materials
    - Filling holes in materials
    - Segmenting Fossils
    - Attempting to segment the cortex in brain imaging

</td>
<td>
<figure>    
<img src="figures/cortex.png" style="height:500px" />
<figcaption>The cortex in brain imaging</figcaption>
</figure>
</td></tr></table>

## Literature / Useful References

- John C. Russ, [The Image Processing Handbook](http://dx.doi.org/10.1201/9780203881095) 

### Models / ROC Curves

- The ROC curve [wikipedia](https://en.wikipedia.org/wiki/Receiver_operating_characteristic)
- Julia Evans [Recalling with Precision](https://www.youtube.com/watch?v=ryZL4XNUmwo)
- [Stripe's Next Top Model](https://github.com/stripe/topmodel)

# Why do we do imaging experiments?

There are different reasons for performing an image experiment. This often depends on in which state you are in your project. 

## Exploratory

In the initial phase, you want to learn what your sample looks like with the chosen modality. Maybe, you don't even know what is in there to see. The explorative type of experiment mostly only allows qualitative conclusions. These conclusions will however help you to formulate better hypotheses for more detailed experiments.

 - To visually, qualitatively examine samples and differences between them
 - No prior knowledge or expectations
 
## To test a hypothesis

When you perform an experiment to test a hypothesis, you already know relatively much about your sample and want make an investigation where you can quantify characteristic features.

Quantitative assessment coupled with statistical analysis
 - Does temperature affect bubble size?
 - Is this gene important for cell shape and thus mechanosensation in bone?
 - Does higher canal volume make bones weaker?
 - Does the granule shape affect battery life expectancy?

## Exploratory - What we are looking at?

In the exploratory analysis we inspect the images to get a first impression of the analysis situation. This inspection may be sufficient to set a diagnosis. In ofther cases is gives us a first glimpse to tell the complexity of the upcoming analytical tasks.

The following figures shows different views of a cell. What we would expect as an illustration and what a micrograph image of a cell sample would look like. It is not always so easy to like the idealized structures with the observed ones in the measured image.
```{figure} figures/Average_prokaryote_cell.png
:width: 8cm
Schematic illustration of a cell.
```
```{figure} figures/Electron-micrograph-of-an-isolated-chief-cell-The-cell-contains-numerous-large-zymogen_W640.jpg
:width: 8cm
Electron micrograph of an isolated chief cell. The cell contains numerous large zymogen.
```

<div class='row'>
    <div class='column'>
<figure>    
<img src="figures/Average_prokaryote_cell.svg" style="height:500px" />
<figcaption><a href="http://en.wikipedia.org/wiki/File:Average_prokaryote_cell-_en.svg">Standard Cell</a></figcaption>
</figure>
    </div>
    <figure>    
<img src="figures/Electron-micrograph-of-an-isolated-chief-cell-The-cell-contains-numerous-large-zymogen_W640.jpg" style="height:500px" />
<figcaption><a href="https://doi.org/10.1083/jcb.65.2.428">Electron micrograph of an isolated endocrine cell.</a></figcaption>
</figure>    
</div>

## To test a hypothesis

We perform an experiment bone to see how big the cells are inside the tissue:

We have performed an experiment that produced heaps of data to analyze. For example a using tomography.

```{figure} figures/tomoimage.png
:width: 12cm
Acquisition workflow to obtain CT slices of a specimen.
```

At the beginning we have 2560 x 2560 x 2160 x 32 bits = 56GB / sample! Then we apply some filtering and preprocessing to prepare the data for analysis. After 20h of computer time we still have 56GB of data (it is however nicer to work with). 

 <img src="figures/tomoimage.png" style="width:60%"> 

$$\begin{array}{c}2560\times{}2560\times{}2160\times{}32~\mbox{ bits}=56~\mbox{ GB/sample}\\ \downarrow \\ \mbox{Filtering and Preprocessing!} \\
\downarrow \\\mbox{Hours of computer time later ... Still 56GB of data, but less noisy}\end{array}$$

<div class="alert alert-block alert-warning">
<center>
    
__Way too much data, we need to reduce__
    
</center>
</div>

## What did we want in the first place?

In a quantitative image analysis task we need to define metrics that relate to stated hypothesis. Some information can be obtained straight from the images. Examples are:

### *Single numbers*:
* Volume _fraction_,
* Cell _count_,
* Average cell _stretch_,
* Cell volume _variability_

> These are all __measurable__ metrics!

In other cases we need to extract image metrics that are proxies for the true information. 

## Why do we perform segmentation?

In model-based analysis every step we perform, simple or complicated is related to an underlying model of the system we are dealing with

- Identify relevant regions in the images
- Many methods are available to solve the segmentation task. 
- Choose wisely... [_Occam's Razor_](http://en.wikipedia.org/wiki/Occams_Razor) is very important here : 

> __The simplest solution is usually the right one__

Advanced methods like a Bayesian, neural networks optimized using genetic algorithms with Fuzzy logic has a much larger parameter space to explore, establish sensitivity in, and must perform much better and be tested much more thoroughly than thresholding to be justified. 
 

The next two lectures will cover powerful segmentation techinques, in particular with unknown data.

## Review: Filtering and Image Enhancement 

This was a noise process which was added to otherwise clean imaging data

In last week's lecture we saw that images rarely are as clean as we would like and that it can, to some degree, be fixed by using filters that suppress noise and artefacts.

```{figure} ../Lecture-03/figures/imperfect_imaging_system.png
:width: 12cm
Measurements rarely produce perfect images.
```

<center><img src="../Lecture-03/figures/imperfect_imaging_system.svg" style="height:300px" align="middle"></center>

$$ I_{measured}(x,y) = I_{sample}(x,y) + \text{Noise}(x,y) $$

- What would the perfect filter be

$$ \textit{Filter} \ast I_{sample}(x,y) = I_{sample}(x,y) $$

<br/>

$$ \textit{Filter} \ast \text{Noise}(x,y) = 0 $$ 

<br/>

$$ \textit{Filter} \ast I_{measured}(x,y) = \textit{Filter} \ast I_{real}(x,y) + \textit{Filter}\ast \text{Noise}(x,y) \rightarrow \bf I_{sample}(x,y) $$

### Review: Filter performance

<div class="alert alert-block alert-success">

What __most filters__ end up doing
    
$$\textit{Filter}\ast{}I_{measured}(x,y)=90\%\,I_{real}(x,y)+10\%\,\text{Noise}(x,y)$$
    
</div> 

Filters are not the solution to all problem, but they can get you much closer to the solution. There are gains and losses in applying a filter. The important result is that the relevant information is amplified without too much blurring and that the noise and artefacts are suppressed compared to the signal. I.e., the SNR should improve by the application of a filter.

<div class="alert alert-block alert-danger">

What __bad filters__ do

$$\textit{Filter} \ast I_{measured}(x,y) = 10\%\, I_{real}(x,y) + 90\%\, \text{Noise}(x,y)$$
    
</div> 

This case could in principle happen when you test filtering strategies and should naturally be rejected. One case which is hard to obtain good SNR is when you apply high-pass filters, because these amplify high frequencies. The noise is mostly more dominant for high frequencies.

# What we get from the imaging modality

To demonstrate what we get from a modality, we load rubber duck radiograph as a toy example.

In [ ]:
%matplotlib inline
from skimage.io import imread
from skimage.color import rgb2gray
import matplotlib.pyplot as plt

fig,ax=plt.subplots(1,figsize=(12,7))
dkimg = imread("figures/duck/normalized.tif")
ax.imshow(dkimg, cmap = 'bone');
ax.set(xticks=[],yticks=[], title='An X-ray radiograph of a rubber duck');

## Qualitative Metrics: What did people use to do?

What comes out of our detector / enhancement process 

In [ ]:
%matplotlib inline
from skimage.io import imread
from skimage.color import rgb2gray
import matplotlib.pyplot as plt

In [ ]:
dkimg = tiff.imread("figures/duck/normalized.tif")
fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize = (15,4))

m_show_obj = ax_img.imshow(dkimg, cmap = 'bone')
cb_obj = fig.colorbar(m_show_obj,ax=ax_img,shrink=0.8)
cb_obj.set_label('Transmission'), ax_img.set_title('Measured image')

ax_hist.hist(dkimg.ravel(),bins=100)
ax_hist.set_xlabel('Transmision value')
ax_hist.set_ylabel('Pixel Count'), ax_hist.set_title('Gray level histogram');


The first qualitative steps are:
- Inspect the raw image
- Modify contrast and brightness to see if new features appear
- Compute the histogram
- Describe what you see

In some cases, this may be sufficient. In this example, you could for example see that the circuit board is not perfectly fitted and would say that this is a faulty product.

### Initial analysis - Identify objects by eye

The first qualitative analysis is mostly done by eye. You look at the image to describe what you see. This first assessment will help you decide how to approach the quantitative analysis task. Here, it is important to think about using words that can be translated into an image processing workflow.

 - Count, 
 - Describe qualitatively: "batteries in the bottom", "solder spots on PCB", "Thin skin"

The role of initial qualitative analysis in the broader context of image analysis, particularly in disciplines that involve significant image interpretation, such as medical imaging, remote sensing, microscopy in biological sciences, and materials science.

#### Initial Qualitative Assessment

**Observation by Eye**: The first step in analyzing an image often involves a simple, yet critical, observation by the human eye. This phase is qualitative, where the analyst uses their experience, intuition, and perceptual abilities to identify patterns, anomalies, features of interest, and overall characteristics of the image. This step does not involve complex algorithms or computational tools but relies on human visual and cognitive skills.

#### Description and Documentation

**Descriptive Analysis**: After observing the image, the analyst describes what they see using precise, descriptive language. This description can include noting patterns, textures, colors, shapes, and any anomalies or features of interest. The choice of words is important; it should be detailed and objective, avoiding vague or subjective terminology as much as possible.

#### Bridge to Quantitative Analysis

**Translating Observations into Workflow**: The qualitative assessment informs the subsequent quantitative analysis. The observations made by eye must be translated into a series of steps or operations that can be performed by image processing and analysis software. This translation requires thinking critically about the descriptors used for features and patterns observed in the image. For instance, if one notices a particular texture, they must consider which computational techniques can quantify that texture—perhaps through edge detection algorithms, Fourier transforms for pattern frequency analysis, or segmentation techniques to isolate regions of interest.

#### Importance of Appropriate Terminology

**Workflow-Friendly Vocabulary**: Using terms that can be directly related to image processing operations or concepts is crucial. For example, describing a region as having a "high contrast" suggests the use of thresholding techniques for segmentation, while noting "fine, repetitive patterns" may lead to employing Fourier analysis or specific filtering techniques. The aim is to use a vocabulary that bridges the gap between qualitative observation and quantitative analysis tools.

#### Decision-Making for Quantitative Analysis

**Guiding the Analysis Approach**: The initial qualitative assessment helps in deciding the most appropriate quantitative analysis techniques. It assists in choosing the right tools, algorithms, and parameters for the analysis. For example, the presence of noise identified during the qualitative phase would influence the decision to apply noise-reduction techniques before any further quantitative analysis.

#### Conclusion

The process described emphasizes the iterative and complementary relationship between qualitative and quantitative analysis in image processing. Starting with a qualitative assessment by eye allows for a more informed, directed, and efficient quantitative analysis. It ensures that the selection of computational techniques is relevant to the specific features and challenges identified in the initial visual inspection, thereby improving the accuracy and relevance of the analysis outcomes. This approach is especially valuable in research and applications where precise and meaningful interpretation of images is critical.


### Morphometrics
 - Trace the outline of the object (or sub-structures)

# Segmentation Approaches

In the introduction lecture we talked about how people approach an image analysis problem depending on their background. This is something that becomes very clear when an image is about to be segmented. 


```{figure} figures/analysis_approaches.png
:width: 8cm
Different approaches to the segmentation task depends on your background.
```


They match up well to the world view / perspective 

<img src="../Lecture-04/figures/analysis_approaches.png" style="height:600px" />

## How to approach the segmentation task

### Model based segmentation
The experimentalists approached the segmenation task based on their experience and knowledge about the samples. This results in a top-down approach and quite commonly based on models fitting the real world, _what we actually can see in the images_. The analysis aims at solving the problems needed to provide answers to the defined hypothesis.

### Algorithmic segmentation approach
The opposite approach is to find and use generalized algorithms that provides the results. This approach is driven by the results as the computer vision and deep learning experts often don't have the knowledge to interpret the data.


<font size="10em">
<table>
<tr><th>
Model-Based        
</th>
<th>
Data scientific approach  
</th></tr>
<tr><td>    
Experimentalist   
</td>
<td>
Computer Vision / Deep Learning    
</td></tr>    
<tr><td valign="top">
Problem-driven<br/>
<ul>
<li>Top-down</li>
<li><b>Reality</b> Model-based</li>
</ul>
</td><td valign="top">
Results-driven    
</td></tr>    
</table>
</font>

## Model-based Analysis


The image formation process is the process to use some kind of excitation or impulse probe a sample. This requires the interaction of the four parts in the figure below.
 
```{figure} ../Lecture-01/figures/image-formation.png
:width: 12cm
The elements of the image formation process.
```

- __Impulses__ Light, X-Rays, Electrons, A sharp point, Magnetic field, Sound wave
- __Characteristics__ Electron Shell Levels, Electron Density, Phonons energy levels, Electronic, Spins, Molecular mobility
- __Response__ Absorption, Reflection, Phase Shift, Scattering, Emission
- __Detection__ Your eye, Light sensitive film, CCD / CMOS, Scintillator, Transducer

<img src="../Lecture-01/figures/image-formation.svg" style="height:500px" />

- Many different imaging modalities:  
    micro-CT to MRI to Confocal to Light-field to AFM. 
- Similarities in underlying equations, but different _coefficients_, _units_, and _mechanism_

$$I_{measured}(\vec{x})=F_{system}\left(I_{stimulus}(\vec{x}),S_{sample}(\vec{x})\right)$$

### Direct Imaging (simple)

In many setups there is un-even illumination caused by incorrectly adjusted equipment and fluctations in power and setups

$$F_{system}(a,b)=a*b$$

$$I_{stimulus}=\textrm{Beam}_{profile}$$
$$S_{system}=\alpha(\vec{x})\longrightarrow\alpha(\vec{x})=\frac{I_{measured}(\vec{x})}{\textrm{Beam}_{profile}(\vec{x})}$$


Let's look at a radiograph where beam profile that is penetrates the sample:

In [ ]:
%matplotlib inline
from skimage.io import imread
from skimage.color import rgb2gray
import matplotlib.pyplot as plt
from skimage.morphology import disk
from scipy.ndimage import zoom
import numpy as np

duck_img      = tiff.imread("figures/duck/neglognorm.tif")
duck_imgn     = tiff.imread("figures/duck/normalized.tif")
beam_img      = tiff.imread("figures/duck/ob.tif")
detector_bias = tiff.imread("figures/duck/dc.tif")
detector_img  = tiff.imread("figures/duck/duck90.tif")  

fig =plt.figure(figsize=(15,6))
ax_img  = plt.subplot2grid(shape=(1,4),  loc=(0, 0))
ax_det  = plt.subplot2grid(shape=(1,4),  loc=(0, 3))
ax_propagation = plt.subplot2grid(shape=(2,4),  loc=(0, 1))
ax_beam = plt.subplot2grid(shape=(2,4),  loc=(1, 1))
ax_bias = plt.subplot2grid(shape=(1,4),  loc=(0, 2))


x,y=np.meshgrid(np.linspace(0,2,200),np.linspace(-1,1,200))

ax_propagation.imshow(np.cos(40*np.sqrt(x**2+y**2))*(0.95<np.abs(np.arctan(x/y))))
ax_propagation.set(title='Propagation',xticks=[],yticks=[])

ax_img.imshow(duck_img,    cmap = 'viridis'); 
ax_img.set(title='Sample Profile',xticks=[],yticks=[])

m,s = (beam_img-detector_bias).mean(), (beam_img-detector_bias).std()
ax_beam.imshow(beam_img-detector_bias, clim=[m-s,m+s], cmap = 'gray'); 
ax_beam.set(title='Beam Profile',xticks=[],yticks=[])

m,s = detector_bias.mean(), detector_bias.std()
ax_bias.imshow(detector_bias, clim=[m-s,m+s], cmap = 'gray'); 
ax_bias.set(title='Detector bias',xticks=[],yticks=[])

m,s = detector_img.mean(), detector_img.std()
ax_det.imshow(detector_img, clim=[m-0.5*s,m+1*s],cmap = 'gray');
ax_det.set(title='Measured image',xticks=[],yticks=[]);

The image formation in this example involves three components that combines into the final measured image. 
- The beam propagation and geometric distortion from the divergent beam. This gives a perspective projection of the sample.
- The beam profile which here has two components. The intensity distribution of the source and the amplification factors of the individual detecto elements. 
- The detector bias. This bias is introduced in the detector to better handle the flutuations caused by the thermal noise in the detector. 

### Profiles across the image

A first qualitative analysis on images of this type is to extract line profiles to see how the transmitted intensity changes across the sample. What we can see in this particular example is that the acquired profile tapers off with the beam intensity. With this in mind, it may come clear to you that you need to normalize the images by the beam profile.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12,3.5))
m,s=detector_img.mean(),detector_img.std()
ax[0].imshow(detector_img,clim=[m-s,m+s]); 
ax[0].axhline(beam_img.shape[1]//2,color='red')
ax[1].plot(beam_img[beam_img.shape[1]//2], label = 'Beam Profile')
rax = ax[1].twinx()
rax.plot(duck_imgn[beam_img.shape[1]//2], color='#2ca02c',label = 'Sample Image')
rax.set_yticks(np.linspace(0,1,6))
rax.set_ylabel("Normalized intensity")
ax[1].plot(detector_img[detector_img.shape[1]//2], label = 'Detector')
ax[1].set_ylabel('Intensity'); ax[1].set_xlabel('Pixel Position');

bbox_props = dict(boxstyle="rarrow", fc=(0.9, 0.9, 0.8), ec="r", lw=1)

t = ax[1].text(250, 7700, "Rubber wall", ha="right", va="top", rotation=45,
            size=8,
            bbox=bbox_props)

bbox_props = dict(boxstyle="larrow", fc=(0.9, 0.9, 0.8), ec="r")
t = ax[1].text(1100, 4000, "Battery", ha="left", va="center", rotation=0,
            size=8,
            bbox=bbox_props)

handles1, labels1 = ax[1].get_legend_handles_labels()
handles2, labels2 = rax.get_legend_handles_labels()

# Combine the handles and labels
handles = handles1 + handles2
labels = labels1 + labels2

# Create a legend on ax1 (or ax2) with the combined handles and labels
ax[1].legend(handles, labels, loc='lower right', fontsize=8);

### Inhomogeneous illumination
Frequently there is a gradient of the beam away from the center (as is the case of a Gaussian beam which frequently shows up for laser systems). 

This can make extracting detail away from the center much harder.

### Absorption Imaging (X-ray, Ultrasound, Optical)

__For absorption/attenuation imaging we use [Beer-Lambert Law](http://en.wikipedia.org/wiki/Attenuation_coefficient)__

$$I_{detector}=\underbrace{I_{source}}_{I_{stimulus}}\underbrace{e^{-\alpha d}}_{S_{sample}}$$

Different components have a different $\alpha$ based on 
- the strength of the interaction between the light 
- and the chemical / nuclear structure of the material

$$I_{sample}(x,y)=I_{source}(x,y)\cdot{}e^{-\int_L\alpha(l)dl}$$

<br/>

$$\alpha=f(N,Z,\sigma,\cdots)$$

The intensity measure at each position (x,y) in the object is the result of the line integral along $L$ through the sample at the position of the detector position. I.e., $L$ is located at x,y and parallel to the z-axis. The attenuation coefficient can vary along the line. 


__For segmentation this model is:__
 - there are 2 (or more) distinct components that make up the image
 - these components are distinguishable by their values (or vectors, colors, tensors, ...)

### A numerical transmission imaging example (1D)
In this example we create a sample with three different materials and the sample thickness x=1.0.

The attenuation coefficient is modelled by random models to give each measurement a realistic spread around the expected value. The attenuation coefficient is rarely an exact value in real materials. There can be fluctuations in density caused by porosity and impurities in the material.


The transmission uses Beer Lambert's law.

<div class="row">
<div class="column">

$$I = I_0 e^{-x\cdot \alpha}$$
    
</div>
<div class="column">

| | Material 1 | Material 2 | Material 3 |
|---|---|---|---|
|$\alpha$ |1.0|2.0|3.0|
|$\sigma_{\alpha}$|0.25|0.25|0.5|
        
</div>
</div>

In [ ]:
I_source = 1.0
d = 1.0
alpha_1 = np.random.normal(1, 0.25, size = 100) # Material 1
alpha_2 = np.random.normal(2, 0.25, size = 100) # Material 2
alpha_3 = np.random.normal(3, 0.50, size = 100) # Material 3

# Here, we use a dataframe to build a table of the transmissions
abs_df = pd.DataFrame([dict(alpha = c_x, material = c_mat) for c_vec, c_mat in zip([alpha_1, alpha_2, alpha_3], 
                       ['material 1', 'material 2', 'material 3']) for c_x in c_vec])

abs_df['I_detector'] = I_source*np.exp(-abs_df['alpha']*d)
abs_df.sample(5)

In the table, you can see that we measure different intensities on the detector depending on the material the beam is penetrating.

#### Plotting measured intensities

Let's now plot the intensities and attenuation coefficients and compare the outcome of our transmission experiment.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2,2, figsize = (10, 10))
for c_mat, c_df in abs_df.groupby('material'):
    ax1.scatter(x = c_df['alpha'], 
                y = c_df['I_detector'], 
                label = c_mat,alpha=0.2)
    ax3.hist(c_df['alpha'], alpha = 0.5, label = c_mat)
    ax2.hist(c_df['I_detector'],  alpha = 0.5, label = c_mat, orientation="horizontal")
ax1.set_xlabel('$\\alpha(x,y)$'); 
ax1.set_ylabel('$I_{detector}(x,y)$')
ax1.legend(); 
ax2.legend(); 
ax2.set_title('Measured intensities')
ax3.legend(loc = 0); 
ax3.set_title('Attenuation coeff distribution')
ax4.axis('off');


The materials are differently represented!

This is thanks to the exponential function in the attenuation law. large alpha values seem to be compressed in the observed attenuation image. 

A further observation in this plot is that there is an overlap between the material distributions which introduces an ambiguity when we want to separate regions of different materials later.

## Flatten a transmission image

- A transmission image can be described by Beer-Lambert's law
- Each image has a bias introduced by the detector

$$T=\frac{I_{Measured}-I_{Bias}}{I_{Illumination}-I_{Bias}} = e^{-\int \alpha(x) dx}$$

In [ ]:
ps.visualize_normalization(detector_img,beam_img,detector_bias,duck_imgn)

_T_ is an image normalized between 0 and 1

$$\begin{cases}T=1 & \mbox{No sample between source and detector} \\0<T<1 & \mbox{A sample attenuates the beam to some degree}\\ T=0 & \mbox{The sample is opaque}\end{cases}$$

The $\alpha$-$I_{detector}$ plot shows the curved exponential behaviour we can expect from Beer Lambert's law. Now, if we look at the histogram, we can see that distribution of attenuation coefficients doesn't really match the measured intensity. In this example, it is even so that the widths of the diffent materials have changed places. Great attenuation coefficient results in little transmission and small attenuation coefficient allow more of the beam to penetrate the sample.

# Example Mammography
Mammographic imaging is an area where model-based absorption imaging is problematic. 

Even if we assume a constant illumination (_rarely_ the case), 

```{figure} figures/lateral-mammogram-of-female-breast-with-tumor-92263689-813095ee469b45eabfc9f5f4747758ed.jpg
:width: 10cm
A mammography image of a breast with a malign region.
```

<div class="row">
<div class="column">
<center><figure><img src="figures/lateral-mammogram-of-female-breast-with-tumor-92263689-813095ee469b45eabfc9f5f4747758ed.jpg" style="height:400px"/></figure></center>
</div>
<div class="column">
    
$$I_{detector}=\underbrace{I_{source}}_{I_{stimulus}}\underbrace{e^{-\alpha d}}_{S_{sample}}$$
    
$$\downarrow$$
    
$$I_{detector}(x,y)=I_{source}~e^{-\int_{0}^{l}\alpha(x,y, z)\,dz}$$

</div></div>

The assumption that the attenuation coefficient, $\alpha$, is constant is not valid. Then you see that the exponent turns into an integral along the probing ray and that $\alpha$ is a function of the position in the sample.

This of course leads ambiguity in the interpretation of what the pixel intensity really means.

## Problems to interpret radiography images
Specifically the problem is related to the inability to separate the 
- $\alpha$ - attenuation
- $d$ - thickness
terms. 

To demonstrate this, we model a basic breast volume as a half sphere with a constant absorption factor:

| | Air | Breast tissue |
|:---:|:---:|:---:|
|$\alpha(x,y,z)$| 0| 0.01 |

$\rightarrow$ The $\int$ then turns into a $\Sigma$ in discrete space




As we only have the two conditions air and tissue. Air is however not attenuating. Therefore, the detector signal reduces to

$$I_{detector}(x,y)=I_{source}~e^{-\alpha_{tissue}~d_{tissue}(x,y)}$$

## Building a breast phantom
The breast is here modelled as a half sphere of constant attenuation coefficient:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from skimage.morphology import ball

# For the 3D rendering
import plotly.offline as py
from plotly.figure_factory import create_trisurf
from skimage.measure import marching_cubes

The half sphere is here created using the structure element function from scikit image morphology. More about this module later. The model is then created as a half segment of a ball with radius 50 pixels.

In [ ]:
breast_mask = ball(50)[:,50:]  # This is our model

We can also create a volume rendering of the ball using the plotly module.

In [ ]:
# just for 3D rendering, don't worry about it
py.init_notebook_mode()
vertices, simplices, _, _ = marching_cubes(breast_mask>0)
x,y,z = zip(*vertices) 
fig = create_trisurf(   x=x, y=y, z=z, 
                        plot_edges=False,
                        simplices=simplices,
                        title="Breast Phantom")

fig.update_layout(width=700, height=500)
py.iplot(fig)

### Transmission image of the breast phantom

Our first step is to simulate a transmission image of the breast. This is done by 
1. Summing the attenuation coefficents times the pixel size.
2. Applying Beer-Lambert's law

This produces a 2D image of the side view of the breast.

In [ ]:
breast_alpha = 1e-2                           # The attenuation coefficient
pixel_size   = 0.1                            # The simulated detector has 1mm pixels
breast_vol   = breast_alpha*breast_mask       # Scale the image intensity by attenuation coefficient
i_detector   = np.exp(-pixel_size*
                       np.sum(breast_vol,axis=2))  # Compute the transmission through the phantom

In [ ]:
fig, (ax_breast,ax_profile,ax_hist) = plt.subplots(1, 3, figsize = (15,6))

b_img_obj = ax_breast.imshow(i_detector, cmap = 'bone_r'); 
plt.colorbar(b_img_obj) ;
ax_breast.set_title('Transmission image (side profile)')
ax_breast.axhline(y=50,color='r')

ax_profile.plot(i_detector[50,:], c='r')
ax_profile.set(title="Transmission profile")
ax_hist.hist(i_detector.flatten());
ax_hist.set(title='Distribution of transmission values (histogram)', xlabel='$I_{detector}$', ylabel='Pixel Count');

bbox_props = dict(boxstyle="rarrow", fc=(0.9, 0.9, 0.8), ec="r", lw=2)

t = ax_hist.text(0.99, 1100, "Background pixels", ha="right", va="center", rotation=0,
            size=15,
            bbox=bbox_props)

bbox_props = dict(boxstyle="larrow", fc=(0.9, 0.9, 0.8), ec="r", lw=2)
t = ax_hist.text(0.94, 400, "Breast pixels", ha="left", va="center", rotation=45,
            size=15,
            bbox=bbox_props)


The histogram shows the distribution of the transmitted intensity. Note here that all image pixels are counted in the histogram. Therefore, you get a great number of counts from the background.

### Compute the thickness
If we know that $\alpha$ is constant we can reconstruct the thickness $d$ from the image:

$$ d = -\log(I_{detector})/\alpha$$

This is only valid because we have air ($\alpha=0$) as the second component in the phantom. Otherwise, if it was a denser material we would have a material mixture.

Now, let's compute the breast thickness from the transmission image:

In [ ]:
breast_thickness = -np.log(i_detector)/breast_alpha # Compute the thickness

In [ ]:
fig, (ax_breast, ax_profile,ax_hist) = plt.subplots(1, 3, figsize = (15,6))

b_img_obj = ax_breast.imshow(breast_thickness, cmap = 'bone'); ax_breast.set_title('Thickness image')
ax_breast.axhline(y=50,color='r')
plt.colorbar(b_img_obj)

ax_profile.plot(breast_thickness[50,:], c='r')
ax_profile.set(title="Thickness profile")

ax_hist.hist(breast_thickness.flatten()) ; 
ax_hist.set(title='Distribution of thickness',xlabel='Breast Thickness ($d$) [cm]', ylabel='Pixel Count');

Now, you can see that the intensity is reversed in the thickness image. An important difference is that we now have a first quantitative value that relates to the shape of the breast. The transmission image did not reveal this.

### Visualizing the thickness

The thickness map appears as a parabula instead of an ellipsoid as in the geometric model. The reason is that the thickness starts at 0 cm and can only increase.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize = (12, 8))
# ax  = fig.gca(projection='3d')
ax = fig.add_subplot(1, 1, 1, projection='3d')
# Plot the surface.
yy, xx = np.meshgrid(np.linspace(0, 1, breast_thickness.shape[1]),
                     np.linspace(0, 1, breast_thickness.shape[0]))
surf = ax.plot_surface(xx, yy, breast_thickness, cmap=plt.cm.copper,
                       linewidth=0, antialiased=False)
ax.view_init(elev = 30, azim = 45)
ax.set_zlabel('Breast Thickness');

## What if alpha is not constant?
We run into problems when the $\alpha$ is no longer constant. 

<div class="row">
    <div class="column23">

- For example if we place a dark lump in the center of the breast. 

- It is __impossible__ to tell if the breast is _thicker_ or if the lump inside is _denser_. 
        
    </div>
    <div class="column13">
        <center><figure><img src="figures/lateral-mammogram-of-female-breast-with-tumor-92263689-813095ee469b45eabfc9f5f4747758ed.jpg" style="height:400px"/></figure></center>
       </div>
    </div>

### Building a new model
For the lump below we can see on the individual slices of the sample that the lesion appears quite clearly and is very strangely shaped.

| | Air | Breast tissue | Lump |
|:---:|:---:|:---:|:---:|
|$\alpha(x,y,z)$| 0| 0.01 | 0.02 |

We model the lump as a sphere with $\alpha$=0.02 in the breast phantom. The lump appears with high contrast on the slices through the model. But we are limited to the summed profile from a transmission image. 

In [ ]:
breast_vol2 = breast_alpha*breast_mask
alump       = 0.02
lump        = ball(10)
breast_vol2[10:31, 5:26,40:61]=np.maximum(breast_vol2[10:31, 5:26,40:61],lump*alump);

In [ ]:
from skimage.util import montage as montage2d
fig, ax1 = plt.subplots(1,1, figsize = (15, 6))
ax1.imshow(montage2d(breast_vol2.swapaxes(0,2).swapaxes(1,2)[::3]).transpose(), 
           cmap = 'bone', vmin = breast_alpha*.8, vmax = breast_alpha*1.2);

### Looking at the thickness again
When we make the projection and apply Beer's Law we see that it appears as a relatively constant region in the image

In [ ]:
i_detector2 = np.exp(-pixel_size*np.sum(breast_vol2,axis=2)) # Compute what the detector sees

In [ ]:
fig, (ax_breast,ax_hist) = plt.subplots(1, 2, figsize = (15,6))

b_img_obj = ax_breast.imshow(i_detector2, cmap = 'bone_r')
ax_breast.set(title="Detector image")
plt.colorbar(b_img_obj)

ax_hist.hist(i_detector.ravel(),alpha=0.3, label = 'Original')
ax_hist.hist(i_detector2.ravel(),alpha=0.3, label = 'With lump')
ax_hist.legend()
ax_hist.set_xlabel('$I_{detector}$')
ax_hist.set_ylabel('Pixel Count');
ax_hist.set(title="Intensity distribution")

bbox_props = dict(boxstyle="larrow", fc=(0.9, 0.9, 0.8), ec="r", lw=2)
t = ax_hist.text(0.94, 700, "Anomaly decrease", ha="left", va="center", rotation=45,
            size=15,
            bbox=bbox_props)
t = ax_hist.text(0.92, 870, "Anomaly increase", ha="left", va="center", rotation=25,
            size=15,
            bbox=bbox_props)


Comparing the histograms of the normal phantom and the phantom with a lump, we see that there is a displacement of the intensites. 1, there is an increase of pixels in the lower end of the histogram. 2, the same number of pixels are lost in the mid section of the histogram.

These are subtle changes, but give an indication that inspires further investgations. 

Using the histogram is however not the usual way to make a diagnosis. It is far too imprecise because there are so many factors involved that can have an impact on the histogram shape. A further issue is that you rarely have the healthy image to compare with. Images are only made when there is a problem. Except in the case when regular monitoring is made.


### An anomaly in the thickness reconstruction

It appears as a region in the thickness reconstruction. 

So we cannot fundamentally from this single image answer:
- is the breast oddly shaped? ($d$ changed)
- or does it have an possible tumor inside of it? ($\alpha$ changed)


In [ ]:
breast_thickness2 = -np.log(i_detector2)/1e-2

In [ ]:
fig, (ax_breast,ax_hist) = plt.subplots(1, 2, figsize = (15,6))
b_img_obj = ax_breast.imshow(breast_thickness2, cmap = 'bone')
plt.colorbar(b_img_obj)


ax_hist.hist(breast_thickness.ravel(), alpha=0.3,label='Original thickness')
ax_hist.hist(breast_thickness2.ravel(), alpha=0.3,label='Thickness w. lump')

ax_hist.set_xlabel('Breast Thickness ($d$)\nIn cm')
ax_hist.set_ylabel('Pixel Count');
ax_hist.legend();

### Looking at the thickness profile with lump

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize = (12, 8))
ax  = fig.add_subplot(1,1,1,projection='3d')

# Plot the surface.
yy, xx = np.meshgrid(np.linspace(0, 1, breast_thickness.shape[1]),
                       np.linspace(0, 1, breast_thickness.shape[0]))
surf = ax.plot_surface(xx, yy, breast_thickness2, cmap=plt.cm.copper,
                       linewidth=0, antialiased=False)
ax.view_init(elev = 30, azim = 60)
ax.set_zlabel('Breast Thickness');

## Ambiguity in interpreting transmission images

<img src="figures/ambiguous_transmission.svg" style="height:500px">


```{figure} figures/ambiguous_transmission.png
:width: 12cm
The same intensity on the detector can have different origins.
```

The thickness/density problem can be illustrated by the example in the figure. The same gray level is build up by a combination of thicknesses multiplied by their corresponding attenuation coefficient. Thus, it can happen that the same intensity is registered for regions of very different dimensionsions. 

### Methods to reduce the ambiguity

The problem can be addressed in two ways:
1. Still use transmission imaging, but __use constant thickness__.

<img src="figures/PlantContainers.png" style="height:400px"/>

2. Change to an imaging method that __provides 3D information__. E.g. computed tomography.


```{figure} figures/PlantContainers.png
:width: 12cm
Reduce the ambiguity by constraining the shape.
```

## Summary of the image formation process

- Images from the detector can mostly _not_ be used directly.
- Normalization is needed.
- The information in transmission images can be difficult to interpret.

# Segmentation

## Where does segmentation get us?

We can convert a decimal value or something even more complicated like 
- 3 values for RGB images,
- a spectrum for hyperspectral imaging, 
- or a vector / tensor in a mechanical stress field

To a single or a few discrete values: 
- usually __True__ or __False__, 
- but for images with many phases it would be each phase, e.g. bone, air, cellular tissue.

__2560 x 2560 x 2160 x 32 bit = 56GB / sample__ $\rightarrow$ 2560 x 2560 x 2160 x **1 bit** = 1.75GB / sample


## Basic segmentation: Applying a threshold to an image
Start out with a simple image of a cross with added noise
$$ I(x,y) = f(x,y) $$

Here, we create a test image with two features embedded in uniform noise; a cross with values in the order of '1' and background with values in the order '0'. The figure below shows the image and its histogram. The histogram helps us to see how the graylevels are distributed which guides the decision where to put a threshold that segments the cross from the background.

In [ ]:
nx = 5; ny = 5
xx, yy   = np.meshgrid(np.arange(-nx, nx+1)/nx*2*np.pi, 
                       np.arange(-ny, ny+1)/ny*2*np.pi)
cross_im = (np.abs(xx*yy)<=(2*np.pi/nx))+1.75*np.random.uniform(-0.25, 0.25, size = xx.shape)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(9,6))
im=ax.imshow(cross_im, cmap = 'hot')
fig.colorbar(im);

## The histogram

The intensity can be described with a probability density function 
$$ P_f(x,y) $$

In [ ]:
fig, ax1 = plt.subplots(1)
ax1.hist(cross_im.ravel(), 20)
ax1.set_title('$P_f(x,y)$'); 
ax1.set_xlabel('Intensity'); 
ax1.set_ylabel('Pixel Count');

## Applying a threshold to an image

By examining the image and probability distribution function, we can _deduce_ that the underyling model is a whitish phase that makes up the cross and the darkish background. This is indicated by the left side (lower intensity) that has the greater number of pixels than the right side (higher intensity).

Applying the threshold is a deceptively simple operation

$$ I(x,y) = 
\begin{cases}
1, & f(x,y)\geq0.40 \\
0, & f(x,y)<0.40
\end{cases}$$


We just compare each if each pixel is less or greater than a constant value, the threshold, and say that pixels brighter that the threshold belong the the cross.

In [ ]:
threshold  = 0.4
thresh_img = cross_im > threshold

In [ ]:
fig, (ax2,ax1) = plt.subplots(1,2,figsize=(15,6))
ax1.matshow(cross_im, cmap = 'hot', extent = [xx.min(), xx.max(), yy.min(), yy.max()])

ax1.plot(xx[np.where(thresh_img)]*0.91, yy[np.where(thresh_img)]*0.91,
         'ks', markerfacecolor = 'green', alpha = 0.5, label = 'threshold', markersize = 20)
ax1.legend();
ax2.hist(cross_im.ravel(), 20)
ax2.set_title('$P_f(x,y)$'); ax1.set_xlabel('Intensity'); ax1.set_ylabel('Pixel Count');

ax2.axvline(x=0.4, color='r');

### Various Thresholds

Finding the correct threshold value is crucial for the outcome of the thresholding operation. A bad choice may attribute too many or too few pixels to the intended item. A basic method to explore this is to try different thresholds and observe the effect. This can easily be done in interactive tools like ImageJ. Below, you see the effect on our cross image.

We can see the effect of choosing various thresholds 

$\gamma\in{}\{0.1,0.26,0.42,0.58,0.74,0.9\}$

In [ ]:
fig, m_axs = plt.subplots(2,3, 
                          figsize = (15, 8),dpi=150)
for c_thresh, ax1 in zip(np.linspace(0.1, 0.9, 6), m_axs.flatten()):
    
    ax1.imshow(cross_im,
               cmap = 'bone', 
               extent = [xx.min(), xx.max(), yy.min(), yy.max()])
    thresh_img = cross_im > c_thresh

    ax1.plot(xx[np.where(thresh_img)]*0.91, yy[np.where(thresh_img)]*0.91, 'rs', alpha = 0.5, label = 'img>%2.2f' % c_thresh, markersize = 15)
    ax1.legend(loc = 1);

In this fabricated example we saw that thresholding can be a very simple and quick solution to the segmentation problem. Unfortunately, real data is often less obvious. The features we want to identify for our qantitative analysis are often obscured be different other features in the image. They may be part of the setup of caused by the acquisition conditions.

# Segmenting Cells

We can peform the same sort of analysis with this image of cells

This time we can derive the model from the basic physics of the system

- The field is illuminated by white light of nearly uniform brightness
- Cells absorb light causing darker regions to appear in the image
- _Lighter_ regions have no cells
- __Darker__ regions have cells

In [ ]:
%matplotlib inline
from skimage.io import imread
import matplotlib.pyplot as plt
import numpy as np

cell_img = imread("figures/Cell_Colony.jpg")

fig, (ax_hist, ax_img) = plt.subplots(1, 2, figsize = (15,6), dpi=120)
ax_hist.hist(cell_img.ravel(), np.arange(255))
ax_obj = ax_img.matshow(cell_img, cmap = 'bone')
plt.colorbar(ax_obj);

The human eye/brain is a fantatic segmentation machine. We can easily see and label the cells in the example image. The computer, however, has a harder time for this seemingly trivial task.

## Trying different thresholds on the cell image

Let's try the same threshold scanning approach as we did with the cross image and apply different thresholds to the cell image.

In [ ]:
from skimage.color import label2rgb
fig, m_axs = plt.subplots(2,3, 
                          figsize = (15, 8), dpi = 150)
for c_thresh, ax1 in zip(np.linspace(100, 200, 6), m_axs.flatten()):
    thresh_img = cell_img < c_thresh     
    ax1.imshow(label2rgb(thresh_img, image = 1-cell_img, bg_label = 0, alpha = 0.4),interpolation='None') # Rgb coding of image and mask
    ax1.set_title('img<%2.2f' % c_thresh)

There is a graylevel gradient in the image! This can be seen in that the segmented cells in the lower right corner are larger than those in the upper left corner. 

The cells regions have smooth edges which means that they are identified with different widths depending on the background intensity.

We can only speculate about the origin of this gradient, but one reason could be bad normalization to the illumnation field in the microscope.

### A closer look at the effect of a background gradient

We saw in the previous example that a gradient can bias the size of the detected items. This effect is mostly undesirable. Let's take a look at what happens.

In this example we reduce the problem to 1D, which would correspond to looking at an intensity profile across the image. The signal consists of several peaks with the same shape and amplitude.
There is not background bias in the first case while there is a linearly increasing background in the second case. We apply the same threshold to both signals to observe what happens.

In [ ]:
import numpy as np
from scipy.signal.windows import gaussian

N = 100
sigma = 10

g = np.tile(gaussian(N, std=sigma),10)
slope = np.linspace(0,0.6,len(g))

fig,axs = plt.subplots(1,3,figsize=(15,3))
axs=axs.ravel()
th = 0.65
axs[0].plot(g)
axs[0].axhline(y=th,color='r')
axs[0].set(title='Image profile',ylim=[-0.1,1.5])
axs[1].plot(g+slope)
axs[1].plot(slope)
axs[1].set(title='Image profile with background slope',ylim=[-0.1,1.5])
axs[1].axhline(y=th,color='r')

axs[2].plot(th<g,label="Flat")
axs[2].plot(th<(g+slope),label="Slope")
axs[2].set(title='Thresholded profile',ylim=[-0.1,1.5])
axs[2].legend(ncol=2);

The last panel shows the resulting segmentation from the two cases. The item size is the same for all items for the unbiased signal. The increases from left to right for the biased signal. The reason is that the threshold cuts the structures lower down where the item signal is wider. 

This effect means that we would introduce a bias or uncertainty in the size measurement if the slope is not corrected. We will see later in this lecture how the slope can be corrected to obtain more reliable measurements.

# Other image types

While scalar images are easiest, it is possible for any type of image
$$ I(x,y) = \vec{f}(x,y) $$

In the first lecture, we already introduced images that represent more than one value per position. Examples are:
- Flow fields - a vector field representing direction and amplitude
- Spectral images - each position represents a spectral response, i.e. vectors of wavelength, intensity tuples. 

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Here, we create an image with vectors to show local orientation and intensities to meassure the streng of a signal.

In [ ]:
nx = 10
ny = 10
xx, yy = np.meshgrid(np.linspace(-2*np.pi, 2*np.pi, nx), 
                      np.linspace(-2*np.pi, 2*np.pi, ny))

intensity_img = 1.5*np.abs(np.cos(xx*yy))/(np.abs(xx*yy)+(3*np.pi/nx))+np.random.uniform(-0.25, 0.25, size = xx.shape)

base_df = pd.DataFrame(dict(x = xx.ravel(), 
                            y = yy.ravel(), 
                            I_detector = intensity_img.ravel()))

base_df['x_vec'] = base_df.apply(lambda c_row: c_row['x']/np.sqrt(1e-2+np.square(c_row['x'])+np.square(c_row['y'])), 1)
base_df['y_vec'] = base_df.apply(lambda c_row: c_row['y']/np.sqrt(1e-2+np.square(c_row['x'])+np.square(c_row['y'])), 1)

base_df.sample(5)

## Looking at colocation histograms

The colocation histogram is a powerful tool to visualize how different components are related to each other. It also called bi-variate histogram. In seaborn, there is the ```pairplot``` which shows colocation histograms for all combinations on the data. The diagonal is the histogram of the individual components.

In [ ]:
import seaborn as sns
sns.pairplot(base_df);

## Vector field plot

The vector field is a common way to visualize vector data. It does however only work for small data sets like in this example, otherwise it will be too cluttered and no relevant information will be visible.

Here we map the table columns as
- x,y: Position
- x_vec, y_vec: Orientation
- I_detector: Color map

In [ ]:
fig, ax1 = plt.subplots(1,1, figsize = (5, 5))
ax1.quiver(base_df['x'], base_df['y'], base_df['x_vec'], base_df['y_vec'], base_df['I_detector'], cmap = 'hot');

Larger vector fields can also be visualized, but then it is common to downsample the signal to allow for a reasonably uncluttered visualization.

## Applying a threshold to vector valued image

A threshold is now more difficult to apply since there are now two distinct variables to deal with. The standard approach can be applied to both
$$ I(x,y) = 
\begin{cases}
1, & \vec{f}_x(x,y) \geq0.25 \text{ and}\\
& \vec{f}_y(x,y) \geq0.25 \\
0, & \text{otherwise}
\end{cases}$$

In [ ]:
thresh_df = base_df.copy()
thresh_df['thresh'] = thresh_df.apply(lambda c_row: c_row['x_vec']>0.25 and c_row['y_vec']>0.25, 1)

In [ ]:
fig, ax1 = plt.subplots(1,1, figsize = (4, 4))
ax1.quiver(thresh_df['x'], thresh_df['y'], thresh_df['x_vec'], thresh_df['y_vec'], thresh_df['thresh']);
ax1.set_xlabel('Position x'); ax1.set_ylabel('Position y');

### Histogram of the vectors
This can also be shown on the joint probability distribution as a bivariate histogram.

The lines here indicate the thresholded vector components.

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (5, 5))
ax.hist2d(thresh_df['x_vec'], thresh_df['y_vec'], cmap = 'viridis'); 
ax.set_title('Tresholded'); 
ax.set_xlabel('$\\vec{f}_x(x,y)$'); 
ax.set_ylabel('$\\vec{f}_y(x,y)$');
ax.vlines(0.25,ymin=0.25,ymax=1,color='red',label='x=0.25');
ax.hlines(0.25,xmin=0.25,xmax=1,color='lightgreen', label='y=0.25');
ax.legend(loc='lower left');

### Applying a threshold
Given the presence of two variables; however, more advanced approaches can also be investigated. 
- For example we can keep only components parallel to the x axis by using the dot product.

$$I(x,y)=
\begin{cases}
1,&|\vec{f}(x,y)\cdot{}\vec{i}|=1\\
0,&\text{otherwise}
\end{cases}$$

### Thresholding orientations

We can tune the angular acceptance by using the fact that the scalar product can be expressed using the angle between the the vectors as 

__Scalar product definition__ 

$$\vec{x}\cdot\vec{y}=|\vec{x}| |\vec{y}| \cos(\theta_{x\rightarrow y}) $$
<br />
<br />

$$I(x,y)=
\begin{cases}
1,&\cos^{-1}(\vec{f}(x,y)\cdot \vec{i}) \leq \theta^{\circ} \\
0,&\text{otherwise}
\end{cases}$$

## Summary of basic thresholding
- Thresholding is the first approach to segmentation.
- The histogram guides the threshold selection.
- Inhomogeneous illumination and noise make the task harder.
- Non-scalar pixel values can also be thresholded.